# Sistem Rekomendasi Produk Makeup — Content-Based Filtering

Notebook ini membangun sistem rekomendasi produk makeup berbasis konten, dengan tahapan:

1. Persiapan lingkungan
2. Data Understanding
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Data Preparation
6. Model Development (Content-Based Filtering)
7. Evaluasi
8. Simpan data untuk aplikasi Streamlit
9. Cek konsistensi dengan `recommender.py` (opsional)

Dataset: [Kaggle — zarasarkar/makeup-insights-customer-reviews](https://www.kaggle.com/datasets/zarasarkar/makeup-insights-customer-reviews), file `cleaned_makeup_products.csv`.

### Hubungan notebook ini dengan aplikasi

```
notebook ini  ──►  products_clean.csv  ──►  folder data/  ──►  streamlit run app.py
```

Logika di notebook ini **sama persis** dengan yang dipakai aplikasi. Di aplikasi, logika itu
dikemas sebagai fungsi di `recommender.py`:

| Bagian notebook | Fungsi di `recommender.py` | Dipakai oleh |
|---|---|---|
| 4. Data Preprocessing | `clean_products()`, `summarize_description()` | `prepare_data.py`, `app.py` |
| 5–6. Preparation & Model | `Recommender.fit()`, `Recommender.recommend()` | `app.py` |
| 7. Evaluasi | `evaluate()` | `evaluate.py`, tab Evaluasi di `app.py` |

## 1. Persiapan Lingkungan

Impor library. Variabel `IN_COLAB` mendeteksi apakah notebook berjalan di Google Colab
atau di komputer sendiri (Jupyter/VS Code).

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pd.set_option("display.max_colwidth", 80)
print("Berjalan di Google Colab:", IN_COLAB)

## 2. Data Understanding

### 2.1 Mengambil dataset

1. Buka halaman dataset di Kaggle, klik **Download**, lalu ekstrak file zip-nya.
2. Jalankan sel di bawah. Di **Google Colab**, akan muncul tombol *Choose Files*:
   pilih file `cleaned_makeup_products.csv`.
3. Jika dijalankan di **komputer sendiri** (Jupyter/VS Code), cukup letakkan
   `cleaned_makeup_products.csv` di folder yang sama dengan notebook ini atau di folder
   aplikasi (`makeup-recsys/`). Sel di bawah akan menemukannya otomatis.

In [ ]:
NAMA_FILE = "cleaned_makeup_products.csv"

# Cari file di folder notebook atau di folder induknya (folder aplikasi).
kandidat = [NAMA_FILE, os.path.join("..", NAMA_FILE)]
DATA_FILE = next((p for p in kandidat if os.path.exists(p)), None)

if DATA_FILE is None:
    if not IN_COLAB:
        raise FileNotFoundError(
            f"{NAMA_FILE} belum ada. Letakkan file tersebut di folder notebook ini."
        )
    _ = files.upload()   # pilih cleaned_makeup_products.csv
    DATA_FILE = NAMA_FILE

print("File data:", DATA_FILE)

### 2.2 Memuat data

Variabel `products_raw` menyimpan data **asli** dan tidak akan diubah sampai tahap preprocessing.

In [ ]:
products_raw = pd.read_csv(DATA_FILE)

print("Jumlah baris          :", len(products_raw))
print("Jumlah produk unik (ID):", products_raw["product_link_id"].nunique())
products_raw.info()

Kolom yang dipakai sistem rekomendasi:

| Kolom | Isi | Peran |
|---|---|---|
| `product_link_id` | ID produk | Kunci unik |
| `product_name`, `brand`, `category` | Nama, merek, kategori | Fitur dan informasi tampilan |
| `description` | Deskripsi halaman produk (hasil scraping) | Fitur teks (setelah diringkas) |
| `pros`, `best_uses` | Kelebihan dan kegunaan menurut pengulas | Fitur teks |
| `price`, `average_rating`, `rating_count` | Harga, rating, jumlah rating | Filter dan pemecah skor seri |
| `product_link` | Tautan produk | Tampilan di aplikasi |

## 3. Exploratory Data Analysis

> Semua analisis di bagian ini memakai **salinan** `eda = products_raw.copy()`, sehingga
> data utama tidak ikut berubah oleh operasi eksplorasi (misalnya `drop_duplicates`).

In [ ]:
eda = products_raw.copy()

fig, ax = plt.subplots(figsize=(10, 5))
eda["category"].fillna("(kosong)").value_counts().plot(
    kind="bar", ax=ax, color="skyblue", edgecolor="black")
ax.set_title("Distribusi Produk per Kategori")
ax.set_xlabel("Kategori")
ax.set_ylabel("Jumlah produk")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
eda["brand"].fillna("(kosong)").value_counts().head(10).plot(
    kind="bar", ax=ax, color="skyblue", edgecolor="black")
ax.set_title("10 Nilai Merek Terbanyak (sebelum dibersihkan)")
ax.set_xlabel("Merek")
ax.set_ylabel("Jumlah produk")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

artefak = eda["brand"].astype(str).str.lower().isin(["ask a question", "find your shade"])
print("Baris dengan merek berupa artefak scraping:", int(artefak.sum()))
print("Baris dengan merek kosong                 :", int(eda["brand"].isna().sum()))

"Ask A Question" dan "Find your shade" **bukan nama merek**, melainkan teks tombol di halaman
produk yang ikut terambil saat scraping. Ini diperbaiki di tahap preprocessing (4.3).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dinilai = eda[pd.to_numeric(eda["rating_count"], errors="coerce") > 0]
axes[0].hist(dinilai["average_rating"], bins=20, color="skyblue", edgecolor="black")
axes[0].set_title("Distribusi Rata-rata Rating (produk yang sudah dinilai)")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Jumlah produk")

axes[1].hist(eda["price"].dropna(), bins=30, color="salmon", edgecolor="black")
axes[1].set_title("Distribusi Harga")
axes[1].set_xlabel("Harga (USD)")
axes[1].set_ylabel("Jumlah produk")

plt.tight_layout()
plt.show()

In [ ]:
# Ringkasan nilai kosong pada kolom yang relevan
kolom = ["product_name", "category", "brand", "price", "description",
         "pros", "best_uses", "average_rating"]
pd.DataFrame({
    "nilai_kosong": eda[kolom].isna().sum(),
    "persen": (eda[kolom].isna().mean() * 100).round(2),
})

Banyak produk tidak memiliki `pros` dan `best_uses`. Produk tersebut **tidak dibuang**;
nilai kosong cukup diisi teks kosong karena produk tetap punya nama dan deskripsi untuk dibandingkan.

## 4. Data Preprocessing

Setiap langkah dicatat di `laporan` supaya jelas berapa baris yang terdampak.
Langkah-langkah ini identik dengan fungsi `clean_products()` di `recommender.py`.

In [ ]:
df = products_raw.copy()
laporan = {"baris_awal": len(df)}

# Kolom opsional yang mungkin tidak ada di versi dataset lain
for kol in ["product_link", "category", "brand", "price", "average_rating",
            "rating_count", "review_count", "pros", "best_uses"]:
    if kol not in df.columns:
        df[kol] = np.nan

### 4.1 Buang baris tanpa nama atau deskripsi

Produk tanpa nama atau deskripsi tidak bisa dibandingkan, sehingga dibuang.

In [ ]:
df = df.dropna(subset=["product_name", "description"])
df["product_name"] = df["product_name"].astype(str).str.strip()
df["description"] = df["description"].astype(str)
laporan["dibuang_tanpa_nama_atau_deskripsi"] = laporan["baris_awal"] - len(df)
print("Dibuang:", laporan["dibuang_tanpa_nama_atau_deskripsi"], "| Sisa:", len(df))

### 4.2 Buang duplikat ID produk

In [ ]:
sebelum = len(df)
df = df.drop_duplicates(subset="product_link_id")
laporan["dibuang_duplikat_id"] = sebelum - len(df)
print("Dibuang:", laporan["dibuang_duplikat_id"], "| Sisa:", len(df))

### 4.3 Perbaiki merek

Deskripsi hasil scraping mengikuti pola:

`... navigate each product image <Merek> <Nama Produk> Item ...`

Jadi merek dapat diambil dari teks di antara `product image` dan nama produk.
Jika gagal, merek diisi `Unknown`.

In [ ]:
MEREK_TIDAK_VALID = {"", "nan", "none", "find your shade", "ask a question"}
UNKNOWN = "Unknown"


def ambil_merek_dari_deskripsi(deskripsi, nama_produk):
    pola = r"product image\s+(.+?)\s+" + re.escape(nama_produk)
    cocok = re.search(pola, deskripsi)
    if not cocok:
        return None
    merek = cocok.group(1).strip()
    if 0 < len(merek) <= 40 and merek.lower() not in MEREK_TIDAK_VALID:
        return merek
    return None


merek_baru, diperbaiki = [], 0
for merek, desk, nama in zip(df["brand"], df["description"], df["product_name"]):
    m = "" if pd.isna(merek) else str(merek).strip()
    if m.lower() in MEREK_TIDAK_VALID:
        hasil = ambil_merek_dari_deskripsi(desk, nama)
        if hasil:
            diperbaiki += 1
        m = hasil or UNKNOWN
    merek_baru.append(m)

contoh_sebelum = df["brand"].copy()
df["brand"] = merek_baru
laporan["merek_diperbaiki_dari_deskripsi"] = diperbaiki
laporan["merek_tetap_unknown"] = int((df["brand"] == UNKNOWN).sum())
print("Merek diperbaiki:", diperbaiki, "| Tetap Unknown:", laporan["merek_tetap_unknown"])

# Tampilkan beberapa contoh perbaikan
berubah = contoh_sebelum.astype(str) != df["brand"].astype(str)
pd.DataFrame({"produk": df.loc[berubah, "product_name"],
              "merek_sebelum": contoh_sebelum[berubah],
              "merek_sesudah": df.loc[berubah, "brand"]}).head(10)

### 4.4 Kategori kosong menjadi `Unknown` (produk tetap dipertahankan)

In [ ]:
df["category"] = df["category"].fillna(UNKNOWN).astype(str).str.strip()
df.loc[df["category"] == "", "category"] = UNKNOWN
laporan["kategori_unknown"] = int((df["category"] == UNKNOWN).sum())
print("Kategori Unknown:", laporan["kategori_unknown"])

### 4.5 Rapikan kolom numerik

Rating `0` dengan jumlah rating `0` berarti produk **belum dinilai**, bukan dinilai buruk.
Nilainya diubah menjadi kosong (`NaN`) agar tidak menurunkan rata-rata.

In [ ]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")
jumlah_rating = pd.to_numeric(df["rating_count"], errors="coerce")
belum_dinilai = (jumlah_rating == 0) & (df["average_rating"] == 0)
df.loc[belum_dinilai, "average_rating"] = np.nan
df["rating_count"] = jumlah_rating.fillna(0).astype(int)
print("Produk belum dinilai:", int(belum_dinilai.sum()))

### 4.6 Gabungkan varian produk yang sama

Produk dengan nama dan merek yang sama (misalnya varian ukuran) disimpan satu saja,
yaitu yang jumlah ratingnya terbanyak.

In [ ]:
sebelum = len(df)
df = (df.sort_values("rating_count", ascending=False)
        .drop_duplicates(subset=["product_name", "brand"]))
laporan["dibuang_varian_nama_merek_sama"] = sebelum - len(df)
print("Dibuang:", laporan["dibuang_varian_nama_merek_sama"], "| Sisa:", len(df))

### 4.7 Ringkas deskripsi

Deskripsi asli memuat teks navigasi halaman, daftar *ingredients* lengkap, dan info pengiriman.
Bagian itu menambah *noise* pada TF-IDF. Yang diambil hanya bagian **Summary** sampai sebelum **How To Use**.

In [ ]:
def ringkas_deskripsi(deskripsi):
    teks = str(deskripsi)
    awal = teks.find(" Summary ")
    if awal != -1:
        teks = teks[awal + len(" Summary "):]
    akhir = teks.find(" How To Use ")
    if akhir == -1:
        m = re.search(r"(?<!Key) Ingredients ", teks)
        akhir = m.start() if m else -1
    if akhir != -1:
        teks = teks[:akhir]
    potong = teks.find(" Shipping & Coupon")
    if potong != -1:
        teks = teks[:potong]
    teks = teks.replace("Conscious Beauty at Ulta Beauty™", " ")
    teks = re.sub(r"\b(Details|Benefits|Key Ingredients|Formulated Without|Research Results)\b",
                  " ", teks)
    return re.sub(r"\s+", " ", teks).strip()


df["summary"] = df["description"].map(ringkas_deskripsi)

i = 0
print("SEBELUM (", len(df["description"].iloc[i]), "karakter):\n", df["description"].iloc[i][:400], "...\n")
print("SESUDAH (", len(df["summary"].iloc[i]), "karakter):\n", df["summary"].iloc[i][:400])

### 4.8 Isi kolom teks yang kosong dan tampilkan laporan

In [ ]:
for kol in ["pros", "best_uses"]:
    df[kol] = df[kol].fillna("").astype(str)

df = df.sort_values("product_link_id").reset_index(drop=True)
laporan["baris_akhir"] = len(df)

print("Jumlah produk setelah preprocessing:", len(df), "\n")
pd.Series(laporan, name="jumlah").to_frame()

## 5. Data Preparation

Setiap produk direpresentasikan sebagai satu teks gabungan (`fitur_teks`) dari:
nama produk + ringkasan deskripsi + *best uses* + *pros*.

Kolom kategori dan merek **tidak** dimasukkan ke teks, melainkan dihitung terpisah di bagian 6.3
agar bobotnya bisa diatur.

In [ ]:
KOLOM_TEKS = ["product_name", "summary", "best_uses", "pros"]
df["fitur_teks"] = df[KOLOM_TEKS].fillna("").astype(str).agg(" ".join, axis=1)

df[["product_name", "brand", "category", "fitur_teks"]].head()

## 6. Model Development — Content-Based Filtering

### 6.1 TF-IDF Vectorizer

Pengaturan yang dipakai:

| Parameter | Nilai | Alasan |
|---|---|---|
| `stop_words` | `"english"` | Buang kata umum (the, and, for) |
| `ngram_range` | `(1, 2)` | Tangkap frasa, misalnya "full coverage", "dark circles" |
| `min_df` | 2 | Abaikan kata yang hanya muncul di satu produk (noise/salah ketik) |
| `max_df` | 0,85 | Abaikan kata yang muncul di hampir semua produk |
| `sublinear_tf` | `True` | Kata yang diulang berkali-kali tidak mendominasi |

In [ ]:
tf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2 if len(df) >= 50 else 1,
    max_df=0.85,
    sublinear_tf=True,
)
tfidf_matrix = tf.fit_transform(df["fitur_teks"])

print("Ukuran matriks TF-IDF:", tfidf_matrix.shape, "(produk x fitur kata/frasa)")
print("Contoh fitur:", list(tf.get_feature_names_out()[:15]))

### 6.2 Cosine Similarity

Kemiripan teks antarproduk dihitung dengan *cosine similarity*. Nilainya 0 (tidak mirip)
sampai 1 (identik).

In [ ]:
cosine_sim_teks = cosine_similarity(tfidf_matrix)
print("Ukuran matriks kemiripan:", cosine_sim_teks.shape)

cosine_sim_df = pd.DataFrame(cosine_sim_teks, index=df["product_name"], columns=df["product_name"])
cosine_sim_df.sample(5, axis=1, random_state=1).sample(8, axis=0, random_state=1).round(3)

### 6.3 Fungsi rekomendasi

Skor akhir adalah rata-rata berbobot tiga komponen:

$$\text{skor} = \frac{w_{teks}\cdot\text{kemiripan teks} + w_{kat}\cdot[\text{kategori sama}] + w_{merek}\cdot[\text{merek sama}]}{w_{teks}+w_{kat}+w_{merek}}$$

`[kategori sama]` bernilai 1 jika kategorinya sama, selain itu 0. Hal yang sama berlaku untuk merek.
Jika ada skor yang sama persis (seri), produk dengan rating lebih tinggi didahulukan.

Bobot bawaan: teks 0,7; kategori 0,3; merek 0. Di aplikasi, bobot ini bisa diatur dengan slider.

In [ ]:
kategori = df["category"].to_numpy()
merek = df["brand"].to_numpy()
rating = df["average_rating"].fillna(0).to_numpy()


def hitung_skor(idx, w_teks=0.7, w_kat=0.3, w_merek=0.0):
    sim_teks = cosine_sim_teks[idx]
    sim_kat = ((kategori == kategori[idx]) & (kategori[idx] != UNKNOWN)).astype(float)
    sim_merek = ((merek == merek[idx]) & (merek[idx] != UNKNOWN)).astype(float)
    total = w_teks + w_kat + w_merek
    if total <= 0:
        w_teks, total = 1.0, 1.0
    return (w_teks * sim_teks + w_kat * sim_kat + w_merek * sim_merek) / total, sim_teks


def rekomendasi_produk(nama_produk, k=5, w_teks=0.7, w_kat=0.3, w_merek=0.0,
                       hanya_merek_lain=False):
    cocok = np.flatnonzero(df["product_name"].to_numpy() == nama_produk)
    if cocok.size == 0:
        raise KeyError(f"Produk '{nama_produk}' tidak ditemukan")
    idx = int(cocok[0])

    skor, sim_teks = hitung_skor(idx, w_teks, w_kat, w_merek)
    kandidat = np.ones(len(df), dtype=bool)
    kandidat[idx] = False                      # jangan rekomendasikan produk itu sendiri
    if hanya_merek_lain:
        kandidat &= merek != merek[idx]

    pilih = np.flatnonzero(kandidat)
    urutan = np.lexsort((-rating[pilih], -skor[pilih]))   # skor turun, lalu rating turun
    top = pilih[urutan[:k]]

    hasil = df.iloc[top][["product_name", "brand", "category", "price", "average_rating"]].copy()
    hasil.insert(0, "skor", skor[top].round(4))
    hasil.insert(1, "kemiripan_teks", sim_teks[top].round(4))
    return hasil.reset_index(drop=True)

### 6.4 Mencoba rekomendasi

Contoh memakai produk "Bio Stick Foundation". Jika produk itu tidak ada di data, dipakai produk pertama.

In [ ]:
nama_contoh = "Bio Stick Foundation"
if nama_contoh not in set(df["product_name"]):
    nama_contoh = df["product_name"].iloc[0]

print("Produk acuan:")
display(df.loc[df["product_name"] == nama_contoh,
               ["product_link_id", "product_name", "brand", "category", "price", "average_rating"]])

print("\nTop-5 rekomendasi:")
rekomendasi_produk(nama_contoh, k=5)

In [ ]:
# Mencari alternatif dari merek lain
rekomendasi_produk(nama_contoh, k=5, hanya_merek_lain=True)

## 7. Evaluasi

Karena tidak ada data "produk mana yang disukai pengguna", relevansi didekati dengan
**kategori yang sama** (proxy). Supaya adil, evaluasi memakai **kemiripan teks saja**
(bobot kategori = 0). Kalau kategori ikut menjadi fitur, nilainya pasti mendekati 1.

| Metrik | Arti |
|---|---|
| Precision@k | Rata-rata proporsi rekomendasi yang kategorinya sama dengan produk acuan |
| Baseline acak | Precision@k yang diharapkan jika rekomendasi diambil acak |
| Lift | Precision@k ÷ baseline. Nilai > 1 berarti lebih baik dari acak |
| Cakupan katalog | Proporsi produk yang pernah muncul di daftar rekomendasi |

In [ ]:
def evaluasi(k=5, w_teks=1.0, w_kat=0.0, w_merek=0.0):
    n = len(df)
    jumlah_per_kat = df["category"].value_counts()
    rating_asli = df["average_rating"].to_numpy(dtype=float)

    presisi, baseline, rating_rekom = [], [], []
    per_kat, muncul = {}, set()

    for idx in range(n):
        skor, _ = hitung_skor(idx, w_teks, w_kat, w_merek)
        skor = skor.copy()
        skor[idx] = -np.inf
        top = np.argsort(-skor, kind="stable")[:k]
        muncul.update(top.tolist())
        r = rating_asli[top]
        rating_rekom.extend(r[~np.isnan(r)].tolist())

        c = kategori[idx]
        if c == UNKNOWN or jumlah_per_kat[c] - 1 < k:
            continue   # kategori tidak diketahui atau terlalu sedikit produk sekategori
        p = float(np.mean(kategori[top] == c))
        presisi.append(p)
        baseline.append((jumlah_per_kat[c] - 1) / (n - 1))
        per_kat.setdefault(c, []).append(p)

    p_mean, b_mean = float(np.mean(presisi)), float(np.mean(baseline))
    ringkasan = {
        "k": k,
        "produk": n,
        "query_dievaluasi": len(presisi),
        f"precision@{k}": round(p_mean, 4),
        "baseline_acak": round(b_mean, 4),
        "lift": round(p_mean / b_mean, 2),
        "cakupan_katalog": round(len(muncul) / n, 4),
        "rating_rata2_rekomendasi": round(float(np.mean(rating_rekom)), 2),
        "rating_rata2_katalog": round(float(np.nanmean(rating_asli)), 2),
    }
    tabel_kat = (pd.DataFrame([{"kategori": c, "jumlah_query": len(v),
                                f"precision@{k}": round(float(np.mean(v)), 4)}
                               for c, v in per_kat.items()])
                   .sort_values(f"precision@{k}", ascending=False).reset_index(drop=True))
    return ringkasan, tabel_kat


hasil_eval = {}
for k in [5, 10]:
    ringkas, per_kategori = evaluasi(k=k)
    hasil_eval[k] = ringkas
    print(f"--- k = {k} ---")
    display(pd.Series(ringkas).to_frame("nilai"))
    display(per_kategori)

### Interpretasi dan keterbatasan

- Bandingkan **Precision@k** dengan **baseline acak**. Lift > 1 berarti fitur teks berhasil
  menangkap kemiripan produk lebih baik daripada memilih acak.
- Precision per kategori menunjukkan kategori mana yang deskripsinya mudah dibedakan
  dan mana yang sering tertukar.
- Keterbatasan: kategori sama hanyalah *proxy* relevansi. Validasi yang lebih kuat membutuhkan
  data interaksi per pengguna (misalnya `cleaned_makeup_reviews.csv` untuk *collaborative filtering*)
  atau uji pengguna.

## 8. Simpan Data untuk Aplikasi Streamlit

File `products_clean.csv` adalah jembatan antara notebook ini dan aplikasi.

- **Colab:** file otomatis terunduh. Salin ke folder `makeup-recsys/data/`.
- **Komputer sendiri:** jika notebook ini berada di `makeup-recsys/notebook/`,
  file langsung tersimpan ke `makeup-recsys/data/products_clean.csv`.

Setelah itu, jalankan aplikasi dari folder `makeup-recsys`:

```
streamlit run app.py
```

In [ ]:
KOLOM_SIMPAN = ["product_link_id", "product_link", "product_name", "brand", "category",
                "price", "average_rating", "rating_count", "review_count",
                "description", "summary", "pros", "best_uses"]

if os.path.exists(os.path.join("..", "app.py")):
    os.makedirs(os.path.join("..", "data"), exist_ok=True)
    PATH_OUTPUT = os.path.join("..", "data", "products_clean.csv")
else:
    PATH_OUTPUT = "products_clean.csv"

df[[k for k in KOLOM_SIMPAN if k in df.columns]].to_csv(PATH_OUTPUT, index=False)
print(f"Disimpan: {PATH_OUTPUT} ({len(df)} produk)")

if IN_COLAB:
    files.download(PATH_OUTPUT)

## 9. Cek Konsistensi dengan `recommender.py` (opsional)

Sel ini hanya berjalan bila notebook dibuka di dalam folder aplikasi (`makeup-recsys/notebook/`).
Tujuannya memastikan hasil notebook **sama persis** dengan modul yang dipakai aplikasi:
produk yang tersisa, merek hasil perbaikan, dan nilai evaluasi.

In [ ]:
import sys

if os.path.exists(os.path.join("..", "recommender.py")):
    sys.path.insert(0, os.path.abspath(".."))
    from recommender import Recommender, clean_products, evaluate

    df_app, laporan_app = clean_products(products_raw)
    sama_produk = set(df_app["product_link_id"]) == set(df["product_link_id"])
    gabung = df_app.merge(df, on="product_link_id", suffixes=("_app", "_nb"))
    sama_merek = (gabung["brand_app"] == gabung["brand_nb"]).all()
    sama_laporan = laporan_app == laporan

    eval_app = evaluate(Recommender.fit(df_app), k=5)
    sama_eval = abs(eval_app["precision_at_k"] - hasil_eval[5]["precision@5"]) < 1e-4

    print("Produk sama          :", sama_produk)
    print("Merek sama           :", bool(sama_merek))
    print("Laporan sama         :", sama_laporan)
    print("Precision@5 sama     :", sama_eval)
    assert sama_produk and sama_merek and sama_laporan and sama_eval, "Ada perbedaan!"
    print("\nNotebook dan aplikasi konsisten.")
else:
    print("recommender.py tidak ditemukan di folder induk; pemeriksaan dilewati.")